# GPU Index Builder — Historical Data

Downloads the **Historical Data** tab from [builder.icx.global](https://builder.icx.global/) for every GPU model and every billing type.

The page is HTTP Basic Auth. Put credentials in `.env` next to this notebook:

```
ICX_BUILDER_USER=your_user
ICX_BUILDER_PASSWORD="your_password"
ICX_BUILDER_URL=https://builder.icx.global/
```

The tab calls:

- `GET /api/gpus` for the model list
- `GET /local/providers` for provider-level daily prices (this is what **Export CSV** downloads)
- `GET /local/summary` for the daily min / max / avg

Billing types match the page: `ON_DEMAND`, `RESERVATION`, `SPOT`, `CUSTOM`.

**Re-run:** open this notebook from the project folder, select the **ICX (.venv)** kernel, and run all cells. The first cell refuses to continue if the interpreter is not `.venv`. Files already in `data/parts/` are skipped. Delete `data/parts/` to download again.

Outputs:

- `data/gpus.csv`
- `data/historical_providers.csv`
- `data/historical_summary.csv`

In [ ]:
import csv
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import pandas as pd
import requests

def find_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    return start

def load_env(path: Path) -> None:
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "'\"":
            value = value[1:-1]
        os.environ.setdefault(key.strip(), value)

ROOT = find_root()
VENV_PYTHON = ROOT / ".venv" / "bin" / "python"
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError(
        "Run this notebook with the project .venv.\n"
        f"Expected: {VENV_PYTHON}\n"
        f"Current:  {sys.executable}"
    )
ENV_PATH = ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f"Missing {ENV_PATH}. Create it with ICX_BUILDER_USER and ICX_BUILDER_PASSWORD.")
load_env(ENV_PATH)

USER = os.environ["ICX_BUILDER_USER"]
PASSWORD = os.environ["ICX_BUILDER_PASSWORD"]
BASE = os.environ.get("ICX_BUILDER_URL", "https://builder.icx.global/").rstrip("/")

DATE_START = "2026-04-01"
DATE_END = date.today().isoformat()
BILLING_TYPES = ["ON_DEMAND", "RESERVATION", "SPOT", "CUSTOM"]
WORKERS = 3

DATA = ROOT / "data"
PARTS = DATA / "parts"
DATA.mkdir(exist_ok=True)
PARTS.mkdir(exist_ok=True)

PROVIDER_COLUMNS = [
    "date", "provider_id", "provider_name", "provider_country", "gpu_model",
    "billing_type", "hourly_per_gpu", "hourly", "gpu_count", "vram_total_gb",
    "availability", "interconnect", "reservation_months", "gpu_mem_bandwidth_gbps",
]
SUMMARY_COLUMNS = [
    "gpu_model", "billing_type", "date", "offering_count", "provider_count",
    "min_price", "max_price", "avg_price", "price_range",
]

print(f"Root: {ROOT}")
print(f"Python: {sys.executable}")
print(f"Range: {DATE_START} → {DATE_END}")
print(f"User: {USER}")

In [ ]:
class AuthError(RuntimeError):
    pass

def make_session() -> requests.Session:
    session = requests.Session()
    session.auth = (USER, PASSWORD)
    session.headers.update({"Accept": "application/json"})
    return session

def get_json(session: requests.Session, path: str, params: dict | None = None) -> dict:
    clean = {k: v for k, v in (params or {}).items() if v not in ("", None)}
    last_error = None
    for attempt in range(1, 5):
        try:
            response = session.get(f"{BASE}{path}", params=clean, timeout=360)
            if response.status_code in (401, 403):
                raise AuthError(f"Authentication failed ({response.status_code}). Check .env.")
            if response.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {response.status_code} for {path}")
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict):
                raise ValueError(f"Expected JSON object from {path}")
            return payload
        except AuthError:
            raise
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"{path} {clean} failed after retries: {last_error}")

def write_csv(path: Path, rows: list[dict], columns: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({column: "" if row.get(column) is None else row.get(column) for column in columns})
    temporary.replace(path)

def part_paths(gpu_id: str, billing: str) -> tuple[Path, Path]:
    stem = f"{gpu_id}__{billing}"
    return PARTS / f"{stem}.providers.csv", PARTS / f"{stem}.summary.csv"

session = make_session()
status = get_json(session, "/local/status")
print(status)

In [ ]:
def list_gpus(session: requests.Session) -> list[dict]:
    models = []
    page = 1
    page_count = 1
    while page <= page_count:
        payload = get_json(session, "/api/gpus", {"page": page, "page_size": 100})
        page_count = int(payload.get("page_count") or 1)
        models.extend(payload.get("data") or [])
        page += 1
    return models

gpus = list_gpus(session)
gpu_catalog = pd.DataFrame(gpus)
gpu_catalog.to_csv(DATA / "gpus.csv", index=False)
print(f"{len(gpus)} GPU models")
gpu_catalog[["id", "name", "vendor", "vram_gb"]].head(10)

In [ ]:
def fetch_pair(gpu_id: str, billing: str) -> tuple[str, str, int, int, float, str]:
    providers_path, summary_path = part_paths(gpu_id, billing)
    if providers_path.exists() and summary_path.exists():
        with providers_path.open(encoding="utf-8") as handle:
            provider_rows = max(sum(1 for _ in handle) - 1, 0)
        with summary_path.open(encoding="utf-8") as handle:
            summary_rows = max(sum(1 for _ in handle) - 1, 0)
        return gpu_id, billing, provider_rows, summary_rows, 0.0, "cached"

    started = time.time()
    local = make_session()
    params = {
        "gpu_model": gpu_id,
        "billing_type": billing,
        "date_start": DATE_START,
        "date_end": DATE_END,
    }
    providers = get_json(local, "/local/providers", params).get("data") or []
    summary_rows = []
    for row in get_json(local, "/local/summary", params).get("data") or []:
        summary_rows.append({"gpu_model": gpu_id, "billing_type": billing, **row})
    write_csv(providers_path, providers, PROVIDER_COLUMNS)
    write_csv(summary_path, summary_rows, SUMMARY_COLUMNS)
    return gpu_id, billing, len(providers), len(summary_rows), time.time() - started, "downloaded"

jobs = [(gpu["id"], billing) for gpu in gpus for billing in BILLING_TYPES]
print(f"{len(jobs)} queries ({len(gpus)} GPUs × {len(BILLING_TYPES)} billing types), {WORKERS} workers")

results = []
errors = []
print_lock = threading.Lock()
started_all = time.time()
done = 0

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(fetch_pair, gpu_id, billing): (gpu_id, billing) for gpu_id, billing in jobs}
    try:
        for future in as_completed(futures):
            gpu_id, billing = futures[future]
            try:
                result = future.result()
            except AuthError:
                pool.shutdown(wait=False, cancel_futures=True)
                raise
            except Exception as exc:
                errors.append({"gpu_model": gpu_id, "billing_type": billing, "error": str(exc)})
                result = None
            done += 1
            elapsed = time.time() - started_all
            remaining = (len(jobs) - done) * (elapsed / done)
            with print_lock:
                if result is None:
                    print(f"[{done}/{len(jobs)}] {gpu_id} {billing} FAILED  eta {remaining/60:.1f} min", flush=True)
                else:
                    results.append(result)
                    _, _, provider_rows, summary_rows, seconds, how = result
                    print(
                        f"[{done}/{len(jobs)}] {gpu_id} {billing} {how} "
                        f"providers={provider_rows} summary_days={summary_rows} "
                        f"{seconds:.1f}s  eta {remaining/60:.1f} min",
                        flush=True,
                    )
    except AuthError:
        raise

if errors:
    pd.DataFrame(errors).to_csv(DATA / "fetch_errors.csv", index=False)
    print(f"{len(errors)} queries failed. See data/fetch_errors.csv")
else:
    error_file = DATA / "fetch_errors.csv"
    if error_file.exists():
        error_file.unlink()
    print("All queries succeeded.")

print(f"Finished in {(time.time() - started_all)/60:.1f} min")

In [ ]:
def concat_parts(suffix: str, destination: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(PARTS.glob(f"*.{suffix}.csv")):
        frame = pd.read_csv(path)
        if not frame.empty:
            frames.append(frame)
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    combined.to_csv(destination, index=False)
    return combined

providers = concat_parts("providers", DATA / "historical_providers.csv")
summary = concat_parts("summary", DATA / "historical_summary.csv")

if not providers.empty:
    providers["date"] = pd.to_datetime(providers["date"])
    names = gpu_catalog.set_index("id")["name"]
    providers.insert(5, "gpu_name", providers["gpu_model"].map(names))
    providers.to_csv(DATA / "historical_providers.csv", index=False)

if not summary.empty:
    summary["date"] = pd.to_datetime(summary["date"])
    names = gpu_catalog.set_index("id")["name"]
    summary.insert(1, "gpu_name", summary["gpu_model"].map(names))
    summary.to_csv(DATA / "historical_summary.csv", index=False)

print(f"Provider rows: {len(providers):,}")
print(f"Summary rows:  {len(summary):,}")
if not providers.empty:
    print(f"Dates: {providers['date'].min().date()} → {providers['date'].max().date()}")
    print(f"GPUs with prices: {providers['gpu_model'].nunique()}")
    print(f"Billing types: {sorted(providers['billing_type'].dropna().unique())}")

counts = (
    providers.groupby(["gpu_model", "billing_type"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
    if not providers.empty
    else pd.DataFrame()
)
counts.to_csv(DATA / "row_counts.csv", index=False)
counts.head(20)